# 02 - GDS Louvain Community Detection (optional)

Group aircraft into **risk communities** by shared fault patterns using the Neo4j Graph Data Science (GDS) library.

## What you'll learn
- Build a virtual co-occurrence graph with Cypher aggregation projection
- Run Louvain community detection on a weighted relationship graph
- Write the results back as a node property and inspect the communities

## Core idea
Two aircraft share a weighted edge if they experienced the same fault type. The more fault types in common, the stronger the edge. Louvain finds groups of aircraft that fail alike, which may benefit from coordinated maintenance.

## Prerequisites
- **01_etl_to_neo4j** has loaded the topology and maintenance chain
- A Neo4j instance with the **GDS plugin** (Aura Professional or higher)

This notebook is an optional branch. It is not required by notebooks 03-05.

## Section 1: Configuration

In [ ]:
%pip install neo4j

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI = ""
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready - {NEO4J_URI}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Run Cypher and return a list of dicts."""
    with driver.session() as session:
        return [dict(r) for r in session.run(cypher, params or {})]

print(run_query("RETURN 'Connected to Neo4j' AS status")[0]["status"])

## Section 2: Explore the fault landscape

Before building the community graph, look at how fault types are distributed across the fleet.

In [ ]:
fault_dist = run_query("""
    MATCH (m:MaintenanceEvent)
    RETURN m.fault AS fault_type,
           count(*) AS occurrences,
           count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) AS critical_count
    ORDER BY occurrences DESC
""")

print(f"{'Fault Type':<30} {'Total':>8} {'Critical':>10}")
print("-" * 52)
for row in fault_dist:
    print(f"{row['fault_type']:<30} {row['occurrences']:>8} {row['critical_count']:>10}")

In [ ]:
model_faults = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    RETURN a.model AS model, m.fault AS fault_type, count(*) AS count
    ORDER BY model, count DESC
""")

current = None
for row in model_faults:
    if row["model"] != current:
        current = row["model"]
        print(f"\n{current}")
        print(f"  {'Fault':<28} {'Count':>6}")
        print(f"  {'-'*36}")
    print(f"  {row['fault_type']:<28} {row['count']:>6}")

## Section 3: Build the fault co-occurrence projection

Build a virtual Aircraft-to-Aircraft graph with Cypher aggregation projection. No `SHARES_FAULT` relationship exists in the database. GDS constructs it on the fly at projection time.

**Edge weight** = number of fault types two aircraft have in common.

> Requires the GDS plugin. Run `RETURN gds.version()` to confirm it is available.

In [ ]:
print(f"GDS version: {run_query('RETURN gds.version() AS v')[0]['v']}")

In [ ]:
dropped = run_query("CALL gds.graph.drop('fault-network', false) YIELD graphName")
print(f"Dropped existing projection: {dropped[0]['graphName']}" if dropped else "No existing projection to drop.")

In [ ]:
# elementId(a1) < elementId(a2) avoids creating duplicate edges in both directions.
# The aggregation form of gds.graph.project returns a single map, so alias it (AS g)
# and read the fields off that map.
result = run_query("""
    MATCH (a1:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m1:MaintenanceEvent)
    MATCH (a2:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m2:MaintenanceEvent)
    WHERE elementId(a1) < elementId(a2) AND m1.fault = m2.fault
    WITH a1, a2, count(DISTINCT m1.fault) AS shared_faults
    RETURN gds.graph.project(
        'fault-network', a1, a2,
        {
            sourceNodeLabels: labels(a1),
            targetNodeLabels: labels(a2),
            relationshipType: 'SHARES_FAULT',
            relationshipProperties: {weight: shared_faults}
        },
        {undirectedRelationshipTypes: ['SHARES_FAULT']}
    ) AS g
""")
proj = result[0]["g"]
print(f"Projection: {proj['nodeCount']} nodes, {proj['relationshipCount']} relationships")

## Section 4: Run Louvain

Three execution modes:
1. `stats` - community count and modularity, before committing
2. `stream` - which aircraft land in which community
3. `write` - persist `fault_community` on each Aircraft node

In [ ]:
stats = run_query("""
    CALL gds.louvain.stats('fault-network', {relationshipWeightProperty: 'weight'})
    YIELD communityCount, modularity, ranLevels
""")[0]
print(f"Community count:  {stats['communityCount']}")
print(f"Modularity score: {round(stats['modularity'], 4)}  (higher = better-separated)")
print(f"Levels run:       {stats['ranLevels']}")

In [ ]:
communities = run_query("""
    CALL gds.louvain.stream('fault-network', {relationshipWeightProperty: 'weight'})
    YIELD nodeId, communityId
    RETURN gds.util.asNode(nodeId).tail_number AS aircraft,
           gds.util.asNode(nodeId).model      AS model,
           gds.util.asNode(nodeId).operator   AS operator,
           communityId
    ORDER BY communityId, aircraft
""")

current = None
for row in communities:
    if row["communityId"] != current:
        current = row["communityId"]
        print(f"\nCommunity {current}")
        print(f"  {'Aircraft':<12} {'Model':<12} {'Operator'}")
        print(f"  {'-'*42}")
    print(f"  {row['aircraft']:<12} {row['model']:<12} {row['operator']}")

In [ ]:
result = run_query("""
    CALL gds.louvain.write('fault-network', {
        writeProperty: 'fault_community',
        relationshipWeightProperty: 'weight'
    })
    YIELD communityCount, modularity, nodePropertiesWritten
""")[0]
print(f"Wrote fault_community to {result['nodePropertiesWritten']} Aircraft nodes")
print(f"Communities: {result['communityCount']}, Modularity: {round(result['modularity'], 4)}")

## Section 5: Inspect the communities

Query back through the maintenance chain to understand what defines each community.

In [ ]:
fault_profile = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community AS community, m.fault AS fault_type, count(*) AS occurrences
    ORDER BY community, occurrences DESC
""")

current = None
for row in fault_profile:
    if row["community"] != current:
        current = row["community"]
        print(f"\nCommunity {current} - top faults:")
        print(f"  {'Fault Type':<28} {'Count':>6}")
        print(f"  {'-'*36}")
    print(f"  {row['fault_type']:<28} {row['occurrences']:>6}")

In [ ]:
composition = run_query("""
    MATCH (a:Aircraft)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community            AS community,
           count(a)                     AS aircraft_count,
           collect(DISTINCT a.model)    AS models,
           collect(DISTINCT a.operator) AS operators
    ORDER BY community
""")
for row in composition:
    print(f"Community {row['community']}: {row['aircraft_count']} aircraft")
    print(f"  Models:    {', '.join(row['models'])}")
    print(f"  Operators: {', '.join(row['operators'])}\n")

**Critical event rate per community** - a pure-graph way to risk-rank communities, no sensor telemetry required.

In [ ]:
critical = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    WHERE a.fault_community IS NOT NULL
    RETURN a.fault_community AS community,
           count(m) AS total_events,
           count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) AS critical_events,
           round(100.0 * count(CASE WHEN m.severity = 'CRITICAL' THEN 1 END) / count(m), 1) AS critical_pct
    ORDER BY critical_pct DESC
""")
print(f"{'Community':>10} {'Events':>8} {'Critical':>10} {'Critical %':>12}")
print("-" * 42)
for row in critical:
    print(f"{row['community']:>10} {row['total_events']:>8} {row['critical_events']:>10} {row['critical_pct']:>11}%")

## Section 6: Clean up

Drop the in-memory projection. The `fault_community` property on Aircraft nodes persists.

In [ ]:
run_query("CALL gds.graph.drop('fault-network', false) YIELD graphName")
driver.close()
print("Projection dropped. fault_community remains on Aircraft nodes.")

## Neo4j console queries

```cypher
// View communities
MATCH (a:Aircraft)
WHERE a.fault_community IS NOT NULL
RETURN a.fault_community AS community, collect(a.tail_number) AS aircraft
ORDER BY community
```

```cypher
// Visualize one community's fault chain
MATCH (a:Aircraft {fault_community: 0})-[:HAS_SYSTEM]->(s:System)
      -[:HAS_COMPONENT]->(c:Component)-[:HAS_EVENT]->(m:MaintenanceEvent)
RETURN a, s, c, m
LIMIT 50
```